# Comparar runs e escolher candidato

POC DEMO — contrato PROVISÓRIO / PENDENTE DE HOMOLOGAÇÃO. Execute em kernel novo. Credenciais somente por ambiente.


Configuração tipada. Pré-requisitos: Docker saudável, seed DEMO, token por ambiente. Entradas: parâmetros abaixo; saídas: tabelas, IDs e relatório JSON/Markdown. Escritas exigem confirmação EXECUTE_DEMO e chave estável; repetir chave igual recupera o mesmo job. Espere READY; erros de dependência exigem verificar docker compose logs.

In [ ]:
import os, json
import pandas as pd
from IPython.display import display, Markdown
from src.workbench.journey_config import Settings
cfg=Settings.load()
from src.workbench.notebook_config import Parameters
from src.workbench.operations_sdk import call, report, await_job
params=Parameters.load()
display(params.model_dump(mode="json"))
display({"state":"READY", "write_enabled":params.execute_write, "tracking":"remote Docker service", "browser_ui":cfg.ui})
evidence={}
NOTEBOOK_ID="06_avaliacao_comparativa"


1. Filtrar por snapshot explícito. Compare apenas a mesma janela; baseline e métricas são demonstrativos.

In [ ]:
from src.workbench.operations_sdk import comparison
records=comparison(cfg,str(params.snapshot_id) if params.snapshot_id else "",limit=500)
table=pd.DataFrame(records);display(table)
evidence["runs"]=records


2. Eleger candidato por menor MAE dentro de um único snapshot/janela, sem aprovar ou promover.

In [ ]:
if not table.empty and "mae" in table and params.snapshot_id:
    groups=table.groupby(["snapshot_id","train_rows","evaluation_rows"],dropna=False)
    candidates=[g.sort_values("mae").iloc[0].to_dict() for _,g in groups if g.mae.notna().any()]
    display(pd.DataFrame(candidates));evidence["candidates"]=candidates
else:
    display({"candidate":"NOT_EVALUATED","reason":"Selecione snapshot e runs com mesma janela"})
evidence["promotion"]="NOT_EXECUTED"


Guardar evidência desta execução e links. Esperado: caminho do relatório, sem credenciais.

In [ ]:
report_status="PARTIAL" if any(isinstance(value,str) and value.startswith("NOT_EXECUTED") for value in evidence.values()) else "PASS"
display(report(NOTEBOOK_ID,evidence,status=report_status,limitations=["Dados DEMO e contrato provisório; regras não avaliadas não são homologação", "Execução sem erro do notebook não implica execução de operações opt-in desativadas"]))
